# Phase 1-1 (Validation Variant): 2022 Sentinel-2 Export — 317 DHS Points

**Purpose:** Export 2022 quarterly Sentinel-2 composites for the 317 DHS-origin
clusters in `prediction_points.csv`. These tiles will be passed through the existing
inference pipeline to compare 2022 vs 2025 imagery performance against known 2022
DHS Wealth Index labels (Option 3 accuracy test).

**Changes from original phase1-1:**
- Source: `prediction_points.csv` (DHS rows only) instead of GEE asset `PH_DHS_GPS`
- Urban_Rural field: from CSV column `Urban_Rural` (U/R from DHS GPS record)
- Output folder: `Sentinel2_2022_DHS_Validation` (separate from training tiles)
- Tile naming: `dhs_val_{PointID}_2022_Q{q_num}` to avoid collision with training tiles
- Quarters: identical 2022 Q1-Q4 dates (unchanged)
- Task count: 317 points × 4 quarters = 1,268 tasks (well within GEE 3,000 limit)

In [ ]:
import ee
import os
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

ee.Authenticate()
ee.Initialize(project='integrated-hawk-485001-k3')

In [ ]:
# ── OUTPUT FOLDER ──────────────────────────────────────────────────────────
# Kept separate from Sentinel2_Training_Data to avoid any overwrite risk.
DRIVE_FOLDER_NAME = 'Sentinel2_2022_DHS_Validation'
drive_folder = f'/content/drive/MyDrive/{DRIVE_FOLDER_NAME}'

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f'Created folder: {drive_folder}')
else:
    print(f'Folder exists: {drive_folder}')

In [ ]:
# ── LOAD AND FILTER DHS POINTS FROM CSV ────────────────────────────────────
# Upload prediction_points.csv to Colab before running this cell, or
# load directly from Drive if you have it there.
#
# Option A — upload interactively:
#   from google.colab import files
#   uploaded = files.upload()  # select prediction_points.csv
#   csv_path = 'prediction_points.csv'
#
# Option B — load from Drive (edit path as needed):
csv_path = '/content/drive/MyDrive/prediction_points.csv'

points_df = pd.read_csv(csv_path)

# Filter to DHS-origin points only
dhs_df = points_df[points_df['source'] == 'DHS'].copy()
dhs_df = dhs_df.reset_index(drop=True)

print(f'Total rows in prediction_points.csv : {len(points_df)}')
print(f'DHS-origin points                   : {len(dhs_df)}')
print(f'Urban / Rural split:')
print(dhs_df['Urban_Rural'].value_counts().to_string())
print(f'\nMissing Latitude  : {dhs_df["Latitude"].isna().sum()}')
print(f'Missing Longitude : {dhs_df["Longitude"].isna().sum()}')
print(f'\nSample rows:')
print(dhs_df[['PointID','Latitude','Longitude','Province','Urban_Rural','DHSCLUST']].head(10).to_string())

In [ ]:
# ── BUILD GEE FEATURE COLLECTION FROM CSV COORDINATES ──────────────────────
# Each point becomes an ee.Feature with a Point geometry and the
# properties needed for buffering and file naming.

features = []
for _, row in dhs_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]),
        {
            'PointID'    : int(row['PointID']),
            'Urban_Rural': str(row['Urban_Rural']),
            'Province'   : str(row['Province']),
            'DHSCLUST'   : float(row['DHSCLUST']) if pd.notna(row['DHSCLUST']) else -1
        }
    )
    features.append(feat)

dhs_fc = ee.FeatureCollection(features)
print(f'GEE FeatureCollection built: {len(features)} features')

In [ ]:
# ── ADAPTIVE BUFFER ─────────────────────────────────────────────────────────
# Identical logic to original phase1-1:
#   Urban (U) → 2,000 m buffer → ~4 km × 4 km bounding box
#   Rural (R) → 5,000 m buffer → ~10 km × 10 km bounding box
#
# Urban_Rural is now sourced directly from the DHS GPS displacement
# record (U/R), not approximated from municipal boundaries.

def adaptive_buffer(feature):
    ur = ee.String(feature.get('Urban_Rural'))
    is_urban = ur.compareTo('U').eq(0)
    radius = ee.Number(ee.Algorithms.If(is_urban, 2000, 5000))
    return feature.buffer(radius).bounds()

dhs_squares = dhs_fc.map(adaptive_buffer)
print('Adaptive buffer applied.')

In [ ]:
# ── CLOUD MASKING (identical to original) ───────────────────────────────────

def mask_s2_clouds(image):
    qa   = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask).divide(10000)

In [ ]:
# ── 2022 QUARTERS (identical to original) ───────────────────────────────────
# These match the quarters used for training — same year, same Sentinel-2
# archive windows, same cloud threshold and band selection.

quarters = {
    1: ('2022-01-01', '2022-03-31'),
    2: ('2022-04-01', '2022-06-30'),
    3: ('2022-07-01', '2022-09-30'),
    4: ('2022-10-01', '2022-12-31')
}

In [ ]:
# ── EXPORT LOOP WITH SKIP LOGIC ─────────────────────────────────────────────
# Naming convention: dhs_val_{PointID}_2022_Q{q_num}
#   - 'val' prefix distinguishes these from training tiles (dhs_{DHSCLUST}_2022_Q{q})
#   - PointID is the key linking back to master_cluster_summary.csv
#
# Expected tasks: 317 × 4 = 1,268 — well within GEE's 3,000 concurrent limit.
# No chunking required.

TASK_LIMIT = 2800  # safety buffer

features_list = dhs_squares.getInfo()['features']
total_tasks   = 0
skipped_tasks = 0
failed_points = []

print(f'Found {len(features_list)} DHS features. Scanning Drive for existing files...')

for feature in features_list:

    if total_tasks >= TASK_LIMIT:
        print(f'Reached safe task limit of {TASK_LIMIT}. Stopping queue.')
        break

    props        = feature['properties']
    point_id     = int(props['PointID'])
    roi_geometry = ee.Geometry.Polygon(feature['geometry']['coordinates'])

    for q_num, (start_date, end_date) in quarters.items():

        task_desc         = f'dhs_val_{point_id}_2022_Q{q_num}'
        expected_filepath = os.path.join(drive_folder, f'{task_desc}.tif')

        # Skip if already exported
        if os.path.exists(expected_filepath):
            skipped_tasks += 1
            continue

        # Build quarterly Sentinel-2 composite
        quarterly_col = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi_geometry)
              .filterDate(start_date, end_date)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
        )

        # Primary: cloud-masked median
        best_layer = (
            quarterly_col
              .map(mask_s2_clouds)
              .select(['B4', 'B3', 'B2', 'B8', 'B11'])
              .median()
        )

        # Fallback: unmasked mosaic (for high cloud-cover quarters)
        backup_layer = (
            quarterly_col
              .select(['B4', 'B3', 'B2', 'B8', 'B11'])
              .mosaic()
              .divide(10000)
        )

        final_img = best_layer.unmask(backup_layer).clip(roi_geometry)

        export_task = ee.batch.Export.image.toDrive(
            image       = final_img,
            description = task_desc,
            folder      = DRIVE_FOLDER_NAME,
            region      = roi_geometry,
            scale       = 10,
            crs         = 'EPSG:3857',
            fileFormat  = 'GeoTIFF'
        )

        export_task.start()
        total_tasks += 1

print(f'\nDone.')
print(f'  Queued  : {total_tasks} new export tasks')
print(f'  Skipped : {skipped_tasks} already-existing tiles')
print(f'  Expected: {len(features_list) * 4} total tiles (317 × 4 quarters)')
print(f'  Remaining after skip: {len(features_list) * 4 - skipped_tasks}')

In [ ]:
# ── MONITOR TASK STATUS ─────────────────────────────────────────────────────
# Run this cell periodically to check how many tasks have completed.
# Do NOT re-run the export loop until all tasks are COMPLETED or FAILED.

import time

def check_tasks(prefix='dhs_val_', max_show=20):
    tasks      = ee.data.getTaskList()
    relevant   = [t for t in tasks if t['description'].startswith(prefix)]
    counts     = {}
    for t in relevant:
        state = t['state']
        counts[state] = counts.get(state, 0) + 1
    print(f'Tasks with prefix "{prefix}":  {len(relevant)} total')
    for state, n in sorted(counts.items()):
        print(f'  {state:<15}: {n}')
    failed = [t['description'] for t in relevant if t['state'] == 'FAILED']
    if failed:
        print(f'\nFailed task descriptions (first {max_show}):')
        for desc in failed[:max_show]:
            print(f'  {desc}')
    return failed

failed = check_tasks()

In [ ]:
# ── VERIFY TILE COUNT ON DRIVE ──────────────────────────────────────────────
# Run after all GEE tasks complete. Confirms how many of the
# 1,268 expected tiles (317 × 4) landed on Drive.

tif_files = [f for f in os.listdir(drive_folder) if f.endswith('.tif')]
print(f'GeoTIFF files in {DRIVE_FOLDER_NAME}: {len(tif_files)}')
print(f'Expected                             : {317 * 4}')
print(f'Missing                              : {317 * 4 - len(tif_files)}')

# Find which PointIDs are incomplete (missing one or more quarters)
from collections import defaultdict
tile_map = defaultdict(set)
for fname in tif_files:
    # Expected format: dhs_val_{PointID}_2022_Q{q}.tif
    parts = fname.replace('.tif', '').split('_')
    try:
        pid = int(parts[2])
        q   = int(parts[-1].replace('Q', ''))
        tile_map[pid].add(q)
    except (IndexError, ValueError):
        print(f'  Unexpected filename format: {fname}')

incomplete = {pid: qs for pid, qs in tile_map.items() if len(qs) < 4}
missing_ids = set(dhs_df['PointID'].astype(int)) - set(tile_map.keys())

print(f'\nPointIDs with < 4 quarters : {len(incomplete)}')
print(f'PointIDs with 0 tiles      : {len(missing_ids)}')

if incomplete:
    print('\nIncomplete PointIDs and their present quarters:')
    for pid, qs in sorted(incomplete.items()):
        print(f'  PointID {pid}: has Q{sorted(qs)}, missing Q{sorted({1,2,3,4}-qs)}')